# 02 – Reactive transport: Acid Mine Drainage from the tailings storage facility

In the previous notebook we transported a conservative tracer from the TSF to understand advection and dispersion. Now we ask: **what happens to the water chemistry?**

Acid Mine Drainage (AMD) forms when sulphide minerals — mainly **pyrite (FeS₂)** — oxidise on contact with oxygen and water:

$$\text{FeS}_2 + \tfrac{15}{4}\text{O}_2 + \tfrac{7}{2}\text{H}_2\text{O} \rightarrow \text{Fe}(\text{OH})_3 + 2\,\text{SO}_4^{2-} + 4\,\text{H}^+$$

AMD from the TSF is highly acidic (pH ≈ 4) and enriched in Fe, Al, and SO₄.  
As it migrates through the aquifer it reacts with the aquifer matrix: dissolving carbonates (calcite, siderite), precipitating ferrihydrite, and acidifying groundwater en route to the groundwater-dependent ecosystem (GDE).

## What `mf6rtm` does

`mf6rtm` couples **MODFLOW 6** (flow + transport) with **PHREEQC** (geochemistry) via the ModflowAPI (BMI):

1. At each transport time step, concentrations computed by GWT are passed to PHREEQC.
2. PHREEQC equilibrates or reacts the chemistry (dissolution, precipitation, speciation).
3. The resulting concentrations are returned to GWT for the next transport step.

Rather than one GWT model per run, `mf6rtm` clones the conservative-tracer GWT from notebook 01 into **N reactive GWT models** — one per PHREEQC component — and couples them all through PHREEQC in a single simulation.

In [ ]:
import os
import shutil
import pyemu
import flopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mf6rtm import utils, mup3d
from herebedragons import prep_bins, specify_tsf_cells, plot_setup, sout_to_array

ws          = 'model/flow_transport'   # GWF + tracer GWT built in notebooks 00 and 01
reactive_ws = 'model/reactive'         # mf6rtm will write the reactive simulation here


if os.path.exists(reactive_ws):
    shutil.rmtree(reactive_ws)
os.makedirs(reactive_ws)

## Load the GWF + conservative-tracer GWT simulation

We load the simulation built in notebooks 00 and 01.  
`mf6rtm` uses the tracer GWT as a **template**: it inherits the grid, solver settings, dispersion, and porosity, then clones it into one reactive GWT model per PHREEQC component.

> Make sure you have run notebooks `00` and `01` at least once so the model files exist in `model/flow_transport/`.

In [ ]:
sim      = flopy.mf6.MFSimulation.load(sim_ws=ws, verbosity_level=0)
gwf      = sim.get_model('gwf')
gwt_name = 'gwt'   # conservative tracer model from notebook 01

pitcells = np.argwhere(gwf.dis.idomain.array[0] == 2)
print('Models in sim:', sim.model_names)
print('Grid:         ', gwf.dis.nlay.data, 'layers ×',
      gwf.dis.nrow.data, 'rows ×', gwf.dis.ncol.data, 'cols')

## Defining the water chemistry

We define **two PHREEQC solutions** that represent the two end-members:

| # | Name | pH | SO₄ (mol/L) | Fe (mol/L) | Description |
|---|------|----|-------------|------------|-------------|
| 1 | Aquifer background | 6.96 | 7.5 × 10⁻³ | 5 × 10⁻⁵ | Neutral, mildly mineralised |
| 2 | Tailings leachate (AMD) | 3.99 | 5.0 × 10⁻² | 3 × 10⁻² | Acidic, high SO₄ & Fe |

Concentrations are in **mol/kg water** (PHREEQC default units).  
Solution 1 is assigned as the initial condition everywhere (`set_ic(1)`).  
Solution 2 will be injected at the TSF boundary (CNC package).

In [ ]:
solutionsdf = pd.DataFrame({
    'aquifer': {
        'pH':    6.96,    'pe':    1.67,
        'C(+4)': 3.94e-30, 'S(6)':  7.48e-3,
        'Fe(+2)':5.39e-5, 'Fe(+3)':2.32e-8,
        'Mn(+2)':4.73e-5, 
        'Ca':    6.92e-3,
        'Mg':    1.96e-3, 'Na':    1.30e-3,
        'K':     6.65e-5, 'Cl':    1.03e-3,
        'Al':    1.27e-7, 'Si':    1.94e-3,
    },
    'tailings': {
        'pH':    3.99,    'pe':    7.69,
        'C(+4)': 4.92e-4, 'S(6)':  5.00e-2,
        'Fe(+2)':3.06e-2, 'Fe(+3)':1.99e-7,
        'Mn(+2)':9.83e-6, 
        'Ca':    1.08e-2,
        'Mg':    9.69e-4, 'Na':    1.39e-3,
        'K':     7.93e-4, 'Cl':    1.19e-4,
        'Al':    4.30e-3, 'Si':    2.08e-3,
    },
})
solutionsdf.index.name = 'component'
display(solutionsdf)

solutions = mup3d.Solutions(utils.solution_df_to_dict(solutionsdf))
solutions.set_ic(1)   # every cell starts with background aquifer chemistry (solution 1)

## Aquifer mineralogy: equilibrium with Calcite

As AMD migrates from the TSF, the aquifer's **Acid Neutralising Capacity (ANC)** determines
whether the plume reaches the GDE or is attenuated first.

The primary ANC mineral is **Calcite** (CaCO₃) — the fastest-dissolving carbonate:

$$\text{CaCO}_3 + 2\,\text{H}^+ \rightarrow \text{Ca}^{2+} + \text{H}_2\text{O} + \text{CO}_2$$

This reaction consumes H⁺ from the AMD, raising pH and precipitating dissolved metals (Fe, Al).
Once the Calcite reserve is exhausted, the pH front breaks through and acidification reaches the receptor.

The table below lists the full mineral assemblage; we keep **Calcite only** for simplicity.
`conc_mol_l` is the initial content in mol/L bulk volume, converted to mol/L pore water via porosity.

In [ ]:
prsity = 0.3   # effective porosity (must match MST package in GWT)

eq_df = pd.DataFrame([
    {'phase': 'Calcite',    'sat_index': 0.0, 'conc_mol_l': 1.95e-6, 'num': 1},
    {'phase': 'Siderite',   'sat_index': 0.0, 'conc_mol_l': 4.22e-3, 'num': 1},
    {'phase': 'Gibbsite',   'sat_index': 0.0, 'conc_mol_l': 2.51e-3, 'num': 1},
    {'phase': 'Fe(OH)3(a)', 'sat_index': 0.0, 'conc_mol_l': 1.86e-3, 'num': 1},
    {'phase': 'Gypsum',     'sat_index': 0.0, 'conc_mol_l': 0.0,     'num': 1},
    {'phase': 'SiO2(a)',    'sat_index': 0.0, 'conc_mol_l': 4.07e-1, 'num': 1},
])

# Keep Calcite only — primary ANC mineral for AMD buffering
eq_df = eq_df[eq_df['phase'].isin(['Calcite'])].reset_index(drop=True)

eq_df['conc_mol_lb'] = [
    utils.concentration_volbulk_to_volwater(v, prsity)
    for v in eq_df['conc_mol_l'].values
]

equilibrium_phases = utils.parse_equilibriums_dataframe(eq_df)
eq_phases = mup3d.EquilibriumPhases(equilibrium_phases)
eq_phases.set_ic(1)   # zone 1 everywhere

display(eq_df[['phase', 'sat_index', 'conc_mol_l', 'conc_mol_lb']])
print('\nEquilibrium phases:', eq_phases.names)

## Create the reactive transport model

`Mup3d.from_mf6()` is the entry-point when you already have a flopy simulation with a GWF + GWT pair.  
It reads the grid dimensions from the GWF model and stores the tracer GWT as a template for cloning.

The working directory (`reactive_ws`) is where all mf6rtm files will be written — a clean copy of the simulation with reactive GWT models replacing the tracer.

In [ ]:
model = mup3d.Mup3d.from_mf6(sim, solutions, name='rtm', gwt_name=gwt_name)
model.set_wd(reactive_ws)   # override the default path (same in practice; explicit is better)

print('Grid:        ', model.nlay, model.nrow, model.ncol, f'({model.nxyz:,} cells)')
print('Working dir: ', model.wd)

## PHREEQC thermodynamic database

The database provides:
- **Equilibrium constants** for all aqueous species, complexes, and mineral reactions
- **RATES blocks** containing BASIC programs for kinetically controlled reactions (Pyrite here)

`set_database()` copies the file into the working directory so the simulation is self-contained.

In [ ]:
database = os.path.join('..', 'asset', 'phreeqc.dat')
model.set_database(database)
print('Database copied from:', model.database)

In [ ]:
model.set_charge_offset(0.1)

## Assign mineral phases to the model

`set_equilibrium_phases()` validates that every mineral name exists in the database PHASES block  
and checks that the initial condition array matches the model grid shape.  
The `fill_missing_minerals` utility (called internally) ensures both minerals are defined in every zone —  
here zone 1 covers the entire domain.

In [ ]:
# model.set_equilibrium_phases(eq_phases)
# print('Equilibrium phases registered:', model.equilibrium_phases.names)

## Initialise PHREEQC

`model.initialize()` does several things:

1. Generates the PHREEQC input script (`phinp.dat`) from the solutions, phases, and database.
2. Creates a PhreeqcRM object and equilibrates all grid cells to their initial chemistry (solution 1 = aquifer background).
3. Converts the equilibrated concentrations into the units and array shape expected by MODFLOW 6 GWT (mol/m³, shaped as `[nlay, nrow, ncol]`).
4. Stores component names (`model.components`) — these become the names of the reactive GWT models.
5. Writes the modflow6 files for each componenet and phreeqcrm hooks

This step can take a minute on the full grid.

In [ ]:
model.initialize(add_charge_flag=False)
print('PHREEQC components:', model.components)

## TSF chemical stress — injecting AMD at the boundary

The tailings storage facility is represented as a **constant-concentration** boundary (CNC-type `ChemStress`):

- The TSF footprint cells are the same 15 cells used in notebook 01.
- Each cell is assigned **solution 2** (tailings AMD chemistry).
- `model.set_chem_stress()` equilibrates the AMD chemistry through PHREEQC and stores the component-by-component concentrations that will feed into the per-component CNC packages in the reactive simulation.

The `ChemStress` with `type='cnc'` tells mf6rtm to create a separate `ModflowGwtcnc` package for each PHREEQC component rather than relying on a GWF stress package auxiliary variable.

In [ ]:
tsf_cells_raw = specify_tsf_cells()               # returns [((0, row, col), conc), ...]
tsf_cellids   = [item[0] for item in tsf_cells_raw]  # just the (layer, row, col) tuples

cs = mup3d.ChemStress('tsf', type='cnc')
cs.set_spd([2] * len(tsf_cellids))   # solution 2 = tailings AMD for every TSF cell
cs.set_cells(tsf_cellids)

model.set_chem_stress(cs)  # equilibrates TSF chemistry and stores per-component concentrations
print(f'TSF: {len(tsf_cellids)} cells, solution 2 (tailings AMD)')

## Write and run the reactive simulation

`model.write_simulation()` does the final assembly:

- Clones the tracer GWT into N reactive GWT models (one per PHREEQC component).
- Each GWT gets component-specific initial conditions (`sconc`) and CNC packages (from the ChemStress).
- The GWF model and TDIS are written unchanged — the same flow field drives the reactive transport.
- PHREEQC files (`phinp.dat`, `mf6rtm.yaml`, `mf6rtm.toml`) are also written.

`model.run()` launches the coupled simulation via the MODFLOW 6 BMI (ModflowAPI) and PhreeqcRM.

In [ ]:
# Copy the MODFLOW 6 shared library — mf6rtm uses ModflowAPI (BMI), not the mf6 executable
prep_bins(reactive_ws, get_only=['libmf6', 'mf6'])

# Clip near-zero concentrations before PHREEQC — prevents Newton failure on fringe cells
# model.config.solver['min_concentration'] = 1e-12

model.write_simulation()

print('\nRunning reactive transport simulation...')

In [ ]:
pyemu.os_utils.run('mf6rtm', cwd=reactive_ws)

## Visualise the reactive plume

Each PHREEQC component has its own `.ucn` output file in the reactive directory.  
We focus on **Fe** (total dissolved iron) and **S** (total sulfate) — the two key AMD indicators.

- **Fe** is generated by pyrite oxidation and precipitates as Fe(OH)₃ at higher pH — a retardation signal.
- **S** (as SO₄²⁻) is conservative at near-neutral pH and travels further — a persistence signal.

Concentrations are in **mol/m³** (the units GWT transports internally).

In [ ]:
# Show available output files
ucn_files = sorted([f for f in os.listdir(reactive_ws) if f.endswith('.ucn')])
print('UCN files:', ucn_files)

# Helper: read last time step of a .ucn file
def read_ucn_last(component, ws=reactive_ws):
    fname = os.path.join(ws, f'{component}.ucn')
    cobj = flopy.utils.HeadFile(fname, text='concentration')
    return cobj, np.array(cobj.get_kstpkper())

# Pick which components to plot (fall back gracefully if names differ)
available = [f.replace('.ucn', '') for f in ucn_files]
to_plot   = [c for c in ['Fe', 'S', 'Al'] if c in available]
print('Plotting:', to_plot)



In [ ]:
import matplotlib.patches as patches

sout      = pd.read_csv(os.path.join(reactive_ws, 'sout.csv'))
times     = sorted(sout['time_d'].unique())
labels    = [f'{t/365:.0f} yr' for t in times]
mg        = gwf.modelgrid
idom      = gwf.dis.idomain.array[0]
cell_size = float(gwf.dis.delr.get_data()[0])

tsf_ids = [c[0] for c in specify_tsf_cells()]
tsf_x   = [mg.xcellcenters[r, c] for (_, r, c) in tsf_ids]
tsf_y   = [mg.ycellcenters[r, c] for (_, r, c) in tsf_ids]
tsf_rect_kw = dict(
    xy=(min(tsf_x) - cell_size/2, min(tsf_y) - cell_size/2),
    width=max(tsf_x) - min(tsf_x) + cell_size,
    height=max(tsf_y) - min(tsf_y) + cell_size,
    lw=1.5, edgecolor='navy', facecolor='steelblue', alpha=0.4,
)

vars_cfg = [
    ('solution_total_molality_Al',                      'pH',             'RdYlGn', (None, None)),
    # ('equilibrium_phases_moles_Calcite',  'Calcite (mol)', 'YlGn',   (None, None)),
]

for col_name, label, cmap, (vmin, vmax) in vars_cfg:
    arrays = [sout_to_array(sout[sout['time_d'] == t], col_name, idom) for t in times]
    if vmin is None:
        vals = np.concatenate([a[np.isfinite(a)].ravel() for a in arrays])
        vmin, vmax = float(vals.min()), float(vals.max())

    fig, axes = plt.subplots(1, 4, figsize=(15, 4), constrained_layout=True)
    fig.suptitle(label, fontsize=12, fontweight='bold')
    for ax, arr, lbl in zip(axes, arrays, labels):
        mapview = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax)
        pc = mapview.plot_array(arr, vmin=vmin, vmax=vmax, cmap=cmap, alpha=0.85)
        mapview.plot_grid(alpha=0.2)
        plt.colorbar(pc, ax=ax, shrink=0.7, label=label)
        ax.add_patch(patches.Rectangle(**tsf_rect_kw))
        mapview.plot_bc('wel-dewater', color = 'red', zorder=10)
        ax.set_title(lbl)
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])
    plt.show()

In [ ]:
import matplotlib.patches as patches

ucn_file = os.path.join(reactive_ws, f'Al.ucn')
cobj     = flopy.utils.HeadFile(ucn_file, text='CONCENTRATION')
kstpkper = np.array(cobj.get_kstpkper())   # [[kstp, kper], ...]
labels   = [f'{t/365:.0f} yr' for t in cobj.get_times()]

mg        = gwf.modelgrid
idom      = gwf.dis.idomain.array[0]
cell_size = float(gwf.dis.delr.get_data()[0])

# Pit rectangle from cell centres
pit_x = [mg.xcellcenters[r, c] for r, c in pitcells]
pit_y = [mg.ycellcenters[r, c] for r, c in pitcells]
pit_rect_kw = dict(
    xy=(min(pit_x) - cell_size/2, min(pit_y) - cell_size/2),
    width=max(pit_x) - min(pit_x) + cell_size,
    height=max(pit_y) - min(pit_y) + cell_size,
    lw=1.2, edgecolor='k', facecolor='none',
)

# # TSF rectangle from cell centres
# tsf_ids = [c[0] for c in specify_tsf_cells(
#     col_center=tsf_col, row_center=tsf_row,
#     half_ncol=tsf_half_c, half_nrow=tsf_half_r,
# )]
tsf_x = [mg.xcellcenters[r, c] for (_, r, c) in tsf_ids]
tsf_y = [mg.ycellcenters[r, c] for (_, r, c) in tsf_ids]
tsf_rect_kw = dict(
    xy=(min(tsf_x) - cell_size/2, min(tsf_y) - cell_size/2),
    width=max(tsf_x) - min(tsf_x) + cell_size,
    height=max(tsf_y) - min(tsf_y) + cell_size,
    lw=1.5, edgecolor='navy', facecolor='steelblue', alpha=0.5,
)

# cmap = mcolors.LinearSegmentedColormap.from_list('amd', ['white', 'gold', 'darkorange', 'firebrick'])
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

for ax, ksp, lbl in zip(axes, kstpkper, labels):
    data = cobj.get_data(kstpkper=tuple(ksp))[0].astype(float)
    data[idom <= 0] = np.nan

    mapview = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax)
    pc = mapview.plot_array(data, vmin=0, vmax=1, cmap=cmap, alpha=0.85)
    mapview.plot_grid(alpha=0.05)
    ax.add_patch(patches.Rectangle(**pit_rect_kw))
    ax.add_patch(patches.Rectangle(**tsf_rect_kw))
    ax.set_title(lbl)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(pc, ax=list(axes), label='AMD tracer (normalised)', shrink=0.7)
plt.show()

In [ ]:
import matplotlib.patches as patches

ucn_file = os.path.join(reactive_ws, 'Cl.ucn')
cobj     = flopy.utils.HeadFile(ucn_file, text='CONCENTRATION')
kstpkper = np.array(cobj.get_kstpkper())
labels   = [f'{t/365:.0f} yr' for t in cobj.get_times()]

mg        = gwf.modelgrid
idom      = gwf.dis.idomain.array[0]
cell_size = float(gwf.dis.delr.get_data()[0])

c0_arr = cobj.get_data(kstpkper=tuple(kstpkper[0]))[0].astype(float)
c0_arr[idom <= 0] = np.nan
c0 = float(np.nanmedian(c0_arr))

pit_x = [mg.xcellcenters[r, c] for r, c in pitcells]
pit_y = [mg.ycellcenters[r, c] for r, c in pitcells]
pit_rect_kw = dict(
    xy=(min(pit_x) - cell_size/2, min(pit_y) - cell_size/2),
    width=max(pit_x) - min(pit_x) + cell_size,
    height=max(pit_y) - min(pit_y) + cell_size,
    lw=1.2, edgecolor='k', facecolor='none',
)
tsf_x = [mg.xcellcenters[r, c] for (_, r, c) in tsf_ids]
tsf_y = [mg.ycellcenters[r, c] for (_, r, c) in tsf_ids]
tsf_rect_kw = dict(
    xy=(min(tsf_x) - cell_size/2, min(tsf_y) - cell_size/2),
    width=max(tsf_x) - min(tsf_x) + cell_size,
    height=max(tsf_y) - min(tsf_y) + cell_size,
    lw=1.5, edgecolor='navy', facecolor='steelblue', alpha=0.5,
)

fig, axes = plt.subplots(1, len(kstpkper), figsize=(5*len(kstpkper), 4), constrained_layout=True)
fig.suptitle(f'Cl  (C₀ − C,  C₀ = {c0:.3f} mol/m³)', fontsize=11, fontweight='bold')

absmax = c0
for ax, ksp, lbl in zip(axes, kstpkper, labels):
    data = c0 - cobj.get_data(kstpkper=tuple(ksp))[0].astype(float)
    data[idom <= 0] = np.nan

    mapview = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax)
    pc = mapview.plot_array(data, vmin=0, vmax=absmax, cmap='YlOrRd', alpha=0.85)
    mapview.plot_grid(alpha=0.05)
    ax.add_patch(patches.Rectangle(**pit_rect_kw))
    ax.add_patch(patches.Rectangle(**tsf_rect_kw))
    ax.set_title(lbl)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(pc, ax=list(axes), label='Cl  C₀ − C  (mol/m³)', shrink=0.7)
plt.show()


## Summary

In this notebook we:

1. **Loaded** the GWF + conservative-tracer GWT simulation from notebook 01.
2. **Defined** two PHREEQC solutions: background aquifer chemistry and AMD tailings leachate.
3. **Set up** Pyrite dissolution kinetics (Williamson & Rimstidt 1994 rate law).
4. **Created** a reactive transport model via `Mup3d.from_mf6()` — which clones the tracer GWT into N component-specific GWT models.
5. **Injected** AMD chemistry at the TSF cells via a CNC-type `ChemStress`.
6. **Ran** the coupled MODFLOW 6 + PHREEQC simulation.
7. **Visualised** the spreading of reactive AMD indicators (Fe, S) through the aquifer.

### What to try next
- Add `EquilibriumPhases` (e.g. Calcite, Siderite) to represent mineral buffering that neutralises the plume.
- Increase `m0` for Pyrite and observe more aggressive acidification.
- Change the TSF location to investigate sensitivity to upgradient distance from the GDE.